# Ders 16 — LIBERO-Plus'ı ayrı bir çalışma zamanında kur, VLANeXt ile "yeniden yazılmış / temiz / boş komut" dene

**Önce:** YENİ bir notebook çalışma zamanı aç (Ders 12'nin runtime'ını yeniden KULLANMA: orada `pip install ./third_party/LIBERO` ile orijinal `libero` paketi kurulu; LIBERO-Plus da aynı adı, `libero`'yu kullanır). Çalışma zamanı türü **T4 GPU**. Kaggle: Accelerator **GPU T4 x2**, internet açık; yolları `/content` → `/kaggle/working` yap.

Akış: (1) donanım + disk → (2) sistem kütüphaneleri → (3) VLANeXt + LIBERO-Plus (pip'e KURMADAN, `PYTHONPATH` ile) → (4) `~/.libero_plus/config.yaml` → (5) yamalar (Ders 12 + Ders 16) → (6) asset'ler (6.4 GB zip) → (7) checkpoint (16.4 GB) → (8) duman testi → (9) görev seçimi → (10) koşullar: `rewritten` (LIBERO-Plus'ın kendi komutu) ve `empty` (boş komut); isteğe bağlı `clean` ve `tr` → (11) özet + video.

Disk bütçesi: ~16.4 (checkpoint) + 6.4 (zip; açınca silinir) + açılmış asset'ler (boyutu bilinmiyor — hücre 6 ölçer) + ~4.5 (Qwen3-VL-2B) GB. Süreler hipotezdir; ölçülenleri chat'e yaz.

In [ ]:
# 1) Donanım + disk: cc (7, 5) = T4; boş disk en az ~40 GB olmalı
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk boş %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))

In [ ]:
# 2) Sistem kütüphaneleri: MuJoCo EGL (COMMON_ISSUES #1) + LIBERO-Plus'ın görüntü bozulmaları için ImageMagick (wand)
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 \
    libexpat1 libfontconfig1-dev libmagickwand-dev > /dev/null && echo "apt ok"
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
# 3) VLANeXt + LIBERO-Plus. LIBERO-Plus'ı pip'e kurmuyoruz: PYTHONPATH ile seçiyoruz (VLANeXt README'nin eval komutu da böyle).
#    Böylece hangi 'libero'nun import edildiği tek bir ortam değişkenine bağlı kalır.
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "VLANeXt ok"
%cd /content/VLANeXt
!mkdir -p third_party && git clone -q https://github.com/sylvestf/LIBERO-plus.git third_party/LIBERO-plus && echo "LIBERO-plus ok"
!git -C /content/VLANeXt log -1 --format='VLANeXt %h %ci' && git -C third_party/LIBERO-plus log -1 --format='LIBERO-plus %h %ci'
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" "bddl==1.0.1" future easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds \
    wand scikit-image scipy 2>&1 | tail -2
# extra_requirements.txt'teki usd-core eval import zincirinde yok (pxr import'u aranıp bulunmadı) → kurmuyoruz.
!pip list 2>/dev/null | grep -i -E "^libero|hf-libero" || echo "pip'te libero paketi yok (doğru)"

In [ ]:
# 4) LIBERO-Plus config'i: AYRI klasör (~/.libero_plus) ve yollar açıkça repo kopyasına.
#    LIBERO ilk import'ta config yoksa input() ile soru sorar → notebook donar; dosyayı önceden yazıyoruz.
import os, yaml, pathlib
PLUS = "/content/VLANeXt/third_party/LIBERO-plus"
lib = f"{PLUS}/libero/libero"
cfg_dir = pathlib.Path("~/.libero_plus").expanduser(); cfg_dir.mkdir(exist_ok=True)
yaml.safe_dump({"benchmark_root": lib, "bddl_files": f"{lib}/bddl_files", "init_states": f"{lib}/init_files",
                "datasets": f"{PLUS}/libero/datasets", "assets": f"{lib}/assets"}, open(cfg_dir / "config.yaml", "w"))
os.environ["LIBERO_CONFIG_PATH"] = str(cfg_dir)      # COMMON_ISSUES #6: boş string OLMAMALI
os.environ["LIBERO_PLUS_ROOT"] = PLUS                # Ders 16 yaması 1
os.environ["PYTHONPATH"] = PLUS
print(open(cfg_dir / "config.yaml").read())

In [ ]:
# 5) Yamalar: Ders 12 (sdpa, mmap, VLANEXT_TASK_IDS) + Ders 16 (json yolu, VLANEXT_INSTR_OVERRIDE, VLANEXT_RUN_TAG, log, np.float_)
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py && curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/vlanext_plus_patch.py
!curl -s -o config/t4_plus_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/t4_plus_config.yaml
!python vlanext_t4_patch.py /content/VLANeXt && python vlanext_plus_patch.py /content/VLANeXt

In [ ]:
# 6) Asset'ler: 6.4 GB zip (HF dataset Sylvest/LIBERO-plus) → libero/libero/assets/. Açtıktan sonra zip'i sil.
import time; t0 = time.time()
!hf download --repo-type dataset Sylvest/LIBERO-plus assets.zip --local-dir {lib}
!cd {lib} && unzip -q -o assets.zip && rm assets.zip
!ls {lib}/assets && du -sh {lib}/assets
print("süre %.1f dk" % ((time.time() - t0) / 60))
for d in ["articulated_objects", "new_objects", "scenes", "stable_hope_objects", "stable_scanned_objects", "textures", "turbosquid_objects"]:
    print(("OK   " if os.path.isdir(f"{lib}/assets/{d}") else "EKSİK"), d)

In [ ]:
# 7) Checkpoint — Ders 12'dekiyle aynı dosya (LIBERO'da eğitildi, LIBERO-Plus'ta hiç görmediği bozulmalarla sınanacak)
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints && ls -lh checkpoints

In [ ]:
# 8) Duman testi (model YOK): doğru 'libero' mu import ediliyor, 2402 görev var mı, görev 988'in komutu ne, env açılıyor mu?
!cd /tmp && python -c "import libero, libero.libero as L; print('libero yolu:', list(libero.__path__)); \
from libero.libero import benchmark; s = benchmark.get_benchmark_dict()['libero_spatial'](); print('n_tasks =', s.n_tasks); \
t = s.get_task(988); print('988:', t.name); print('language:', repr(t.language)); print('init:', s.get_task_init_states(988).shape)"
!cd /tmp && python -c "from libero.libero import benchmark, get_libero_path; from libero.libero.envs import OffScreenRenderEnv; import os; \
s = benchmark.get_benchmark_dict()['libero_spatial'](); t = s.get_task(988); \
env = OffScreenRenderEnv(bddl_file_name=os.path.join(get_libero_path('bddl_files'), t.problem_folder, t.bddl_file), camera_heights=128, camera_widths=128); \
env.seed(0); env.reset(); o = env.set_init_state(s.get_task_init_states(988)[0]); print('env ok, agentview', o['agentview_image'].shape); env.close()"

In [ ]:
# 9) Görev seçimi: temel görev 0'ın (between the plate and the ramekin) 46 Language varyantından en kolay 3'ü
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/select_plus_tasks.py
!python select_plus_tasks.py {PLUS} --suite libero_spatial --base 0 --category Language --pick 3 --json secim.json
import json; sel = json.load(open("secim.json")); IDS = sel["task_ids"]; CLEAN = sel["clean_language"]; print(IDS, "|", CLEAN)

In [ ]:
# 10) Koşullar. Her koşul = modeli yeniden yükleyen ayrı bir eval (~3 dk yükleme + 3 episode).
#     rewritten : LIBERO-Plus'ın kendi (LLM ile yeniden yazılmış) komutu        → override YOK
#     empty     : boş komut                                                      → VLANEXT_INSTR_OVERRIDE=""
#     clean     : (isteğe bağlı) orijinal LIBERO komutu, aynı sahne + aynı init  → override = CLEAN
#     tr        : (isteğe bağlı) Ders 15 veri setindeki sp0_tr cümlesi
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/parse_plus_log.py
import subprocess, glob, os
TR = "tabak ile ramekinin arasındaki siyah kaseyi al ve tabağın üstüne koy"   # datasets/01-libero-spatial-tr-mini, sp0_tr
def run(cond, override=None, lang="en"):
    env = dict(os.environ, VLANEXT_TASK_IDS=IDS, VLANEXT_RUN_TAG="_" + cond, CUDA_VISIBLE_DEVICES="0", MUJOCO_EGL_DEVICE_ID="0")
    env.pop("VLANEXT_INSTR_OVERRIDE", None)
    if override is not None: env["VLANEXT_INSTR_OVERRIDE"] = override
    with open(f"stdout_{cond}.txt", "w") as f:
        subprocess.run(["python", "-m", "scripts.libero_plus_bench_eval", "--config", "config/t4_plus_config.yaml"],
                       env=env, stdout=f, stderr=subprocess.STDOUT)
    logs = sorted(glob.glob(f"checkpoints/libero_spatial_checkpoint_*_libero_plus_{cond}*/eval.log"))
    print(cond, "→", logs[-1] if logs else "LOG YOK, stdout_" + cond + ".txt dosyasının sonuna bak")
    if logs:
        r = subprocess.run(["python", "parse_plus_log.py", logs[-1], "--run-id", "vlanext_plus_t4_" + cond,
                            "--condition", cond, "--lang", lang, "--csv", "results/runs.csv"], capture_output=True, text=True)
        print(r.stdout, r.stderr)

run("rewritten")
run("empty", override="", lang="none")

In [ ]:
# 10b) İsteğe bağlı (vaktin varsa): aynı 3 sahnede temiz EN ve Türkçe komut
run("clean", override=CLEAN)
run("tr", override=TR, lang="tr")

In [ ]:
# 11) Özet + videolar. 3 episode'luk oranlar bir 'sonuç' değildir (Ders 12: 3/3'ün Wilson %95 aralığı 43.8–100) — boru hattı kanıtıdır.
import pandas as pd
df = pd.read_csv("results/runs.csv"); print(df.groupby("condition")[["n_episodes"]].sum().join(df.groupby("condition")["success_rate"].mean()))
from IPython.display import Video, display
vids = sorted(glob.glob("checkpoints/*_libero_plus_*/*.mp4")); print(*vids, sep="\n")
display(Video(vids[0], embed=True, width=384))

## Sonra
- `results/runs.csv`, dört `eval.log` ve bir başarısız episode videosunu indir → `projects/04-libero-plus-kurulum/` (şablondan aç).
- README'ye yaz: asset'lerin açılmış boyutu (hücre 6), her hücrenin süresi, duman testindeki `libero yolu` satırı, `rewritten` / `empty` (varsa `clean` / `tr`) episode sonuçları.
- Ders 17'de bu 46 yeniden yazımın türlerini (R1 dikkat dağıtma, R2 sağduyu betimleme, R3 akıl yürütme zinciri) kodda ve veride ayıracağız.